In [1]:
from dissipative_algorithms.hamiltonians import *
from dissipative_algorithms.one_ham_with_ckt import *

In [2]:
import numpy as np
for _old, _new in (("string_", "bytes_"), ("unicode_", "str_"),
                   ("float_", "float64"), ("complex_", "complex128")):
    if not hasattr(np, _old):
        setattr(np, _old, getattr(np, _new))

In [3]:
Ham = HamiltonianBuilder().build_molecule(
    [
    ("H", (0.0, 0.0, 0.0)),
    ("H", (1, 0.0, 0)),
    ("H", (1,0,1))
],
        basis="sto-3g",
        multiplicity=2,
        charge=0,
        active_indices=None,
        ancilla_index=0,
        n_active_electrons=None,
        n_active_orbitals=None,
        use_casscf_orbitals=True,
        max_dense_qubits=12,
        use_procrustes_gauge=False,
        align_reference=None,
        fix_spin_ss=None,
        compute_dense=False,
        spectral_width_method="sparse",
)

[spectral] width=2.974680 via Lanczos in 0.0 s (no dense matrix)


In [4]:
Ham.system_qubits

[cirq.LineQubit(1),
 cirq.LineQubit(2),
 cirq.LineQubit(3),
 cirq.LineQubit(4),
 cirq.LineQubit(5),
 cirq.LineQubit(6)]

In [5]:
type(Ham)

dissipative_algorithms.hamiltonians.HamiltonianData

In [15]:
SquareWaveFilter

dissipative_algorithms.one_ham_with_ckt.SquareWaveFilter

In [6]:
filter_terms =  SquareWaveFilter().get_coefficients(L=2,K=400,tau=0.1,A=1.0,window="lanczos",p=1, drop_zeros=False,eps=1e-15, sign=+1)

In [7]:
pairs = [(i, j) for orbs in ([0,1,2], [3,4,5])
         for n, i in enumerate(orbs) for j in orbs[n + 1:]]
weights = [1.0] * len(pairs)

In [8]:
pairs

[(0, 1), (0, 2), (1, 2), (3, 4), (3, 5), (4, 5)]

In [9]:
weights

[1.0, 1.0, 1.0, 1.0, 1.0, 1.0]

In [10]:
W_list, resources = build_W_circuits_only(
    system_qubits=Ham.system_qubits,
    H_int=Ham.interaction,
    anc=Ham.ancilla,
    tau=filter_terms["tau"],
    filter_terms=filter_terms["filter_terms"],
    T_filter=filter_terms["T_filter"],
    pairs=pairs,
    weights=weights,
    N_applications=100,
    reps_per_W=100,
    trotter_steps_H=1,
    rng_seed=123,
    final_rank=4,
    keep_circuits="all",
)

[trotter] propagator built once in 0.0 s (128 moments); reused for all 100 W blocks
[filter] 201 active terms -> 402 propagator insertions per W
[W sampling] 100 applications -> 6 unique sampled jump-pair circuits
[unique W 1/6] 52567 moments, 135884 ops, 1.8 s
[unique W 2/6] 52668 moments, 135884 ops, 1.3 s
[unique W 3/6] 51759 moments, 135884 ops, 1.1 s
[unique W 4/6] 52567 moments, 135884 ops, 1.2 s
[unique W 5/6] 52062 moments, 135884 ops, 1.3 s
[unique W 6/6] 52062 moments, 135884 ops, 1.5 s
[geometry total] width=7 depth=5232965 ops=13588400


In [11]:
W_list[0][1]

cirq.Moment(
    cirq.PhasedISwapPowGate(phase_exponent=0.25, exponent=-1.0).on(cirq.LineQubit(4), cirq.LineQubit(5)),
)

In [14]:
type(W_list[0])

cirq.circuits.circuit.Circuit

In [31]:
from pyLIQTR.utils.resource_analysis import estimate_resources

In [32]:
#p = estimate_resources(W_list)
#p

In [33]:
q = estimate_resources(W_list[0])
q

{'LogicalQubits': 7, 'T': 6469592, 'Clifford': 13005541}

In [34]:
r = estimate_resources(W_list[0][1])
r

{'T': 0, 'Clifford': 1}

In [35]:
print(type(W_list))
print(type(W_list[0]))
print(type(W_list[0][1]))

<class 'list'>
<class 'cirq.circuits.circuit.Circuit'>
<class 'cirq.circuits.moment.Moment'>


In [36]:
#W = W_list[0]

#W_flat = cirq.Circuit(cirq.decompose(W))

In [37]:
order  = [r["unique_key"] for r in resources]
labels = {k: n for n, k in enumerate(dict.fromkeys(order))}
print("".join(str(labels[k]) for k in order))

from collections import Counter
print(Counter(order))                   

0012234432113213400240301322451352510342024025402521314013021253524313312110031403115150433211110144
Counter({'(4, 5)': 24, '(0, 1)': 18, '(3, 4)': 18, '(3, 5)': 17, '(1, 2)': 14, '(0, 2)': 9})


In [38]:
labels

{'(3, 5)': 0, '(4, 5)': 1, '(0, 1)': 2, '(3, 4)': 3, '(1, 2)': 4, '(0, 2)': 5}